In [17]:

import csv
import json
import logging
import os
import random
import re
import sys
import time
import warnings
from collections import Counter
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
import frontmatter                      
from dotenv import load_dotenv           
from IPython.display import Markdown, display
import torch
import chromadb                          
from rank_bm25 import BM25Okapi          
from sentence_transformers import CrossEncoder, SentenceTransformer 
from google import genai
from google.genai import types  
import gradio as gr
from importlib.metadata import PackageNotFoundError, version
print("All imports successful")

All imports successful


In [20]:
load_dotenv()   


@dataclass(frozen=True)
class Config:
    """Single source of truth for every setting in the project."""
    # Data
    kb_dir: Path = Path("ceylon_shield_knowledge_base")
    eval_csv: Path = Path("shieldbot_evaluation_set.csv")
    log_file: Path = Path("shieldbot_log.csv")

   
    embed_model: str = "BAAI/bge-small-en-v1.5"
    rerank_model: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"
    llm_model: str = os.getenv("SHIELDBOT_LLM", "gemini-2.5-flash")

  
    max_chunk_chars: int = 1500

  
    candidates: int = 20      
    top_k: int = 5           
    rrf_k: int = 60         


    default_mode: str = "customer"   
    seed: int = 42

    def __post_init__(self):
        """Fail fast, with a clear message, if a setting is wrong."""
        if self.default_mode not in ("customer", "staff"):
            raise ValueError(f"default_mode must be 'customer' or 'staff', got {self.default_mode!r}")
        if self.top_k > self.candidates:
            raise ValueError("top_k cannot be larger than candidates")
        for path in (self.kb_dir, self.eval_csv):
            if not path.exists():
                raise FileNotFoundError(
                    f"'{path}' not found. The notebook is running from:\n  {Path.cwd()}\n"
                    "Open the shieldbot folder in VS Code and check the file is directly inside it."
                )


CFG = Config()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
    raise EnvironmentError("GOOGLE_API_KEY is missing. Add it to the .env file and restart the kernel.")



def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(CFG.seed)
warnings.filterwarnings("ignore", category=FutureWarning)

if torch.cuda.is_available():
    DEVICE = "cuda"
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"




logging.basicConfig(level=logging.WARNING,
                    format="%(asctime)s | %(levelname)-7s | %(message)s",
                    datefmt="%H:%M:%S", force=True)
log = logging.getLogger("shieldbot")
log.setLevel(logging.INFO)    

def pkg_version(name: str) -> str:
    try:
        return version(name)
    except PackageNotFoundError:
        return "n/a"


def check_llm_model(model_name: str) -> bool:
    """Ask Google which models this key can use, and confirm ours is one of them."""
    try:
        client = genai.Client(api_key=GOOGLE_API_KEY)
        available = {m.name.removeprefix("models/") for m in client.models.list()}
        return model_name in available
    except Exception as exc:                      # network or key problem
        log.warning("Could not verify the Gemini model: %s", exc)
        return False


print("🛡️  ShieldBot environment")
print("─" * 48)
print("✅ API key loaded (.env)")
print("✅ Knowledge base found:", CFG.kb_dir)
print("✅ Evaluation CSV found:", CFG.eval_csv)
print("ℹ️  Markdown files:", len(list(CFG.kb_dir.rglob("*.md"))), "(includes the README)")
print("✅" if check_llm_model(CFG.llm_model) else "⚠️ ", f"Gemini model available: {CFG.llm_model}")
print("ℹ️  Compute device:", DEVICE)
print("─" * 48)
print("Python", sys.version.split()[0])
print(" | ".join(f"{p} {pkg_version(p)}" for p in
                 ["chromadb", "rank-bm25", "sentence-transformers", "torch", "gradio", "google-genai"]))
print(f"Run started: {datetime.now(timezone.utc):%Y-%m-%d %H:%M} UTC")

🛡️  ShieldBot environment
────────────────────────────────────────────────
✅ API key loaded (.env)
✅ Knowledge base found: ceylon_shield_knowledge_base
✅ Evaluation CSV found: shieldbot_evaluation_set.csv
ℹ️  Markdown files: 27 (includes the README)
✅ Gemini model available: gemini-2.5-flash
ℹ️  Compute device: cpu
────────────────────────────────────────────────
Python 3.13.14
chromadb 1.5.9 | rank-bm25 0.2.2 | sentence-transformers 6.3.0 | torch 2.15.0 | gradio 6.32.0 | google-genai 3.6.1
Run started: 2026-11-26 06:19 UTC


In [21]:
REQUIRED_FIELDS = ("doc_id", "title", "status", "audience")
VALID_STATUS = {"active", "superseded"}
VALID_AUDIENCE = {"public", "internal"}


def load_documents(kb_dir: Path) -> list[dict]:
    """Read every policy .md file: YAML front matter becomes metadata, the rest is the body."""
    documents, seen_ids = [], set()
    for path in sorted(kb_dir.rglob("*.md")):
        post = frontmatter.load(path)
        if "doc_id" not in post.metadata:        # e.g. README.md has no metadata
            log.info("Skipped (no metadata): %s", path.name)
            continue

        meta = dict(post.metadata)
        missing = [f for f in REQUIRED_FIELDS if not meta.get(f)]
        if missing:
            raise ValueError(f"{path.name}: missing metadata fields {missing}")
        if meta["status"] not in VALID_STATUS:
            raise ValueError(f"{path.name}: unexpected status {meta['status']!r}")
        if meta["audience"] not in VALID_AUDIENCE:
            raise ValueError(f"{path.name}: unexpected audience {meta['audience']!r}")
        if meta["doc_id"] in seen_ids:
            raise ValueError(f"{path.name}: duplicate doc_id {meta['doc_id']!r}")
        seen_ids.add(meta["doc_id"])

        documents.append({
            "file": path.name,
            "folder": path.parent.name,
            "metadata": meta,
            "body": post.content,
        })
    return documents


docs = load_documents(CFG.kb_dir)
eval_df = pd.read_csv(CFG.eval_csv)

print(f"Documents loaded: {len(docs)}")
print(f"Evaluation questions: {len(eval_df)}   columns: {list(eval_df.columns)}")
print("Status:  ", dict(Counter(d["metadata"]["status"] for d in docs)))
print("Audience:", dict(Counter(d["metadata"]["audience"] for d in docs)))
print("Question types:", dict(Counter(eval_df["test_type"])))

# Do the source documents named in the evaluation set match our documents?
known = {str(d["metadata"]["doc_id"]).lower() for d in docs} | {d["file"].lower().removesuffix(".md") for d in docs}
named = set()
for value in eval_df["source_documents"].dropna():
    for part in re.split(r"[;,|\n]+", str(value)):
        name = part.strip().lower().rsplit("/", 1)[-1].removesuffix(".md")
        if name and name not in ("none", "n/a", "nan", "-"):
            named.add(name)
unmatched = sorted(named - known)
print("Source names in the evaluation set:", len(named), "| not matching any document:", unmatched or "none")

display(pd.DataFrame([{
    "folder": d["folder"],
    "file": d["file"],
    "product": d["metadata"].get("product"),
    "status": d["metadata"]["status"],
    "audience": d["metadata"]["audience"],
    "effective": d["metadata"].get("effective_date"),
} for d in docs]))

11:49:38 | INFO    | Skipped (no metadata): README.md


Documents loaded: 26
Evaluation questions: 20   columns: ['id', 'question', 'expected_answer', 'source_documents', 'test_type']
Status:   {'active': 25, 'superseded': 1}
Audience: {'public': 24, 'internal': 2}
Question types: {'Version conflict': 1, 'Fact lookup': 6, 'Table reasoning': 2, 'Table lookup': 3, 'Multi-document': 3, 'Cross-product disambiguation': 1, 'Guardrail - internal data': 1, 'Guardrail - out of scope': 1, 'Multi-document / trap': 1, 'Guardrail - financial advice': 1}
Source names in the evaluation set: 16 | not matching any document: ['agent_handbook.md (internal)']


,folder,file,product,status,audience,effective
0,00_company,branch_directory.md,general,active,public,2026-06-01
1,00_company,company_overview.md,general,active,public,2026-01-01
2,00_company,contact_and_service_hours.md,general,active,public,2026-01-01
3,01_motor,motor_premium_table_2026.md,motor,active,public,2026-01-01
4,01_motor,motorshield_comprehensive_policy_2026.md,motor,active,public,2026-01-01
5,01_motor,motorshield_third_party_policy_2026.md,motor,active,public,2026-01-01
6,02_health,health_premium_table_2026.md,health,active,public,2026-01-01
7,02_health,health_shield_plus_policy_2025.md,health,superseded,public,2025-01-01
8,02_health,health_shield_plus_policy_2026.md,health,active,public,2026-01-01
9,03_life,life_premium_table.md,life,active,public,2026-01-01


In [22]:
HEADING_RE = re.compile(r"^(#{1,6})\s+(.*)$")


def split_into_sections(body: str) -> list[dict]:
    """Split a document at markdown headings and keep the heading path of each section."""
    sections, stack, lines = [], [], []
    in_code = False

    def flush():
        text = "\n".join(lines).strip()
        if text:
            path = " > ".join(title for _, title in stack) or "Introduction"
            sections.append({"section": path, "text": text})
        lines.clear()

    for line in body.splitlines():
        if line.strip().startswith("```"):
            in_code = not in_code
        match = None if in_code else HEADING_RE.match(line)
        if match:
            flush()
            level, title = len(match.group(1)), match.group(2).strip()
            while stack and stack[-1][0] >= level:
                stack.pop()
            stack.append((level, title))
        lines.append(line)
    flush()
    return sections


def split_blocks(text: str) -> list[str]:
    """Split text into blocks where each markdown table and each paragraph is one block."""
    blocks, current, current_is_table = [], [], None

    def close():
        if current:
            blocks.append("\n".join(current))
            current.clear()

    for line in text.splitlines():
        if not line.strip():
            close()
            current_is_table = None
            continue
        is_table = line.strip().startswith("|")
        if current and is_table != current_is_table:
            close()
        current.append(line)
        current_is_table = is_table
    close()
    return blocks


def split_long_section(text: str, max_chars: int) -> list[str]:
    """Break a long section into pieces without ever cutting a table in half."""
    if len(text) <= max_chars:
        return [text]
    pieces, current = [], ""
    for block in split_blocks(text):
        if current and len(current) + len(block) + 2 > max_chars:
            pieces.append(current)
            current = ""
        current = f"{current}\n\n{block}" if current else block
    if current:
        pieces.append(current)
    return pieces


def build_chunks(documents: list[dict], max_chars: int) -> list[dict]:
    """Turn every document into chunks that keep their document metadata and section title."""
    result = []
    for d in documents:
        meta = d["metadata"]
        doc_id, title = str(meta["doc_id"]), str(meta["title"])
        product = str(meta.get("product", ""))
        n = 0
        for sec in split_into_sections(d["body"]):
            for i, piece in enumerate(split_long_section(sec["text"], max_chars)):
                if i > 0:
                    piece = f'(continued) {sec["section"]}\n\n{piece}'
                result.append({
                    "id": f"{doc_id}::{n}",
                    "text": piece,
                    "embed_text": f'[{title} | {product} | {sec["section"]}]\n{piece}',
                    "metadata": {
                        "doc_id": doc_id,
                        "title": title,
                        "product": product,
                        "doc_type": str(meta.get("doc_type", "")),
                        "effective_date": str(meta.get("effective_date", "")),
                        "status": str(meta["status"]),
                        "audience": str(meta["audience"]),
                        "section": sec["section"],
                        "file": d["file"],
                    },
                })
                n += 1
    return result


chunks = build_chunks(docs, CFG.max_chunk_chars)

sizes = [len(c["text"]) for c in chunks]
print(f"Total chunks: {len(chunks)}")
print("By status:  ", dict(Counter(c["metadata"]["status"] for c in chunks)))
print("By audience:", dict(Counter(c["metadata"]["audience"] for c in chunks)))
print(f"Chunk size: shortest {min(sizes)} | median {int(np.median(sizes))} | longest {max(sizes)} characters")

per_doc = Counter(c["metadata"]["file"] for c in chunks)
display(pd.DataFrame(per_doc.most_common(), columns=["file", "chunks"]))

sample = next((c for c in chunks if "|" in c["text"] and "---" in c["text"]), None)
if sample:
    print("─" * 60)
    print("Example table chunk:\n")
    print(sample["embed_text"])
    

Total chunks: 140
By status:   {'active': 133, 'superseded': 7}
By audience: {'public': 131, 'internal': 9}
Chunk size: shortest 23 | median 238 | longest 1341 characters


,file,chunks
0,faq_general.md,11
1,motorshield_comprehensive_policy_2026.md,9
2,health_shield_plus_policy_2026.md,9
3,home_protect_policy_2026.md,8
4,claims_overview_procedure.md,8
5,health_shield_plus_policy_2025.md,7
6,travel_guard_policy_2026.md,7
7,motor_claims_procedure.md,7
8,company_overview.md,6
9,life_secure_plans.md,6


────────────────────────────────────────────────────────────
Example table chunk:

[Branch Directory | general | Branch Directory]
# Branch Directory

All branches are **closed on Sundays, Poya days and public holidays**.

| Branch | Address | Phone | Weekdays (Mon–Fri) | Saturday |
|---|---|---|---|---|
| Colombo 03 (Head Office) | No. 245, Galle Road, Colombo 03 | 0117 000 100 | 8:30 AM – 5:00 PM | 9:00 AM – 1:00 PM |
| Negombo | No. 52, Main Street, Negombo | 0317 000 200 | 8:30 AM – 4:30 PM | 9:00 AM – 1:00 PM |
| Kandy | No. 18, Dalada Veediya, Kandy | 0817 000 300 | 8:30 AM – 4:30 PM | 9:00 AM – 1:00 PM |
| Galle | No. 7, Wakwella Road, Galle | 0917 000 400 | 8:30 AM – 4:30 PM | 9:00 AM – 1:00 PM |
| Kurunegala | No. 101, Colombo Road, Kurunegala | 0377 000 500 | 8:30 AM – 4:30 PM | 9:00 AM – 12:30 PM |
| Matara | No. 33, Anagarika Dharmapala Mawatha, Matara | 0417 000 600 | 8:30 AM – 4:30 PM | Closed |
| Jaffna | No. 64, Hospital Road, Jaffna | 0217 000 700 | 8:30 AM – 4:30 PM |

In [23]:
SEP_RE = re.compile(r"^\|[\s:|-]*-{3,}[\s:|-]*$")


def table_blocks(text: str) -> list[list[str]]:
    """Return every markdown table found in the text, each as a list of its lines."""
    blocks, current = [], []
    for line in text.splitlines():
        if line.strip().startswith("|"):
            current.append(line.strip())
        elif current:
            blocks.append(current)
            current = []
    if current:
        blocks.append(current)
    return blocks


def run_structure_tests(documents: list[dict], chunk_list: list[dict]) -> list[str]:
    """Check chunk integrity and that no table was split. Returns a list of problems."""
    problems = []

    ids = [c["id"] for c in chunk_list]
    if len(ids) != len(set(ids)):
        problems.append("duplicate chunk ids")
    if any(not c["text"].strip() for c in chunk_list):
        problems.append("empty chunk found")
    if any(not c["metadata"]["section"] for c in chunk_list):
        problems.append("chunk without a section title")

    for d in documents:
        doc_id = str(d["metadata"]["doc_id"])
        doc_chunks = [c for c in chunk_list if c["metadata"]["doc_id"] == doc_id]
        source_tables = table_blocks(d["body"])
        chunk_tables = [t for c in doc_chunks for t in table_blocks(c["text"])]

        if not doc_chunks:
            problems.append(f"{d['file']}: no chunks produced")
        if len(source_tables) != len(chunk_tables):
            problems.append(f"{d['file']}: {len(source_tables)} tables in source, {len(chunk_tables)} in chunks")
        if any(not any(SEP_RE.match(line) for line in t) for t in chunk_tables):
            problems.append(f"{d['file']}: a table lost its header row")
        for src in source_tables:
            if src not in chunk_tables:
                problems.append(f"{d['file']}: a table was altered or split")
                break

        for c in doc_chunks:
            if c["metadata"]["status"] != d["metadata"]["status"] or c["metadata"]["audience"] != d["metadata"]["audience"]:
                problems.append(f"{d['file']}: chunk metadata differs from document metadata")
                break

    return problems


problems = run_structure_tests(docs, chunks)

total_tables = sum(len(table_blocks(d["body"])) for d in docs)
docs_with_tables = sum(1 for d in docs if table_blocks(d["body"]))
print(f"Tables in the source documents: {total_tables} (in {docs_with_tables} documents)")
print(f"Problems found: {len(problems)}")
for p in problems:
    print("  -", p)
print("✅ All structure tests passed" if not problems else "⚠️ Send me this output")

superseded_files = sorted({c["metadata"]["file"] for c in chunks if c["metadata"]["status"] != "active"})
internal_files = sorted({c["metadata"]["file"] for c in chunks if c["metadata"]["audience"] == "internal"})
print("Files that must be hidden because they are superseded:", superseded_files)
print("Files that must be hidden in customer mode (internal):", internal_files)

Tables in the source documents: 42 (in 23 documents)
Problems found: 0
✅ All structure tests passed
Files that must be hidden because they are superseded: ['health_shield_plus_policy_2025.md']
Files that must be hidden in customer mode (internal): ['agent_handbook.md', 'fraud_awareness_guidelines.md']
